# 🚀 Pricer V2 - Fine-Tuning + RAG Integration

This improved version adds the "Evaluation Layer":
- **Tester Suite**: Automated calculation of RMSLE, hits, and average error.
- **Prediction Chat Layer**: Color-coded feedback on model performance.
- **RAG-Shot-Prompting**: Combining retrieved examples with fine-tuned inference for maximum accuracy.
- **Scatter Plot Visualization**: Ground Truth vs. Model Estimate.

In [ ]:
import os
import re
import math
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, GenerationConfig
from peft import PeftModel
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from datasets import load_dataset
from dotenv import load_dotenv

load_dotenv(override=True)

HF_USER = os.getenv("HF_USERNAME", "ukweh")
BASE_MODEL = "unsloth/Llama-3.2-3B-Instruct"
FINETUNED_MODEL = f"{HF_USER}/pricer-v2-optim" # Adjust to your actual model path

## 1. Professional Evaluation Suite

We implement a robust Tester class to track model performance.

In [ ]:
class PricerTester:
    def __init__(self, predictor, data, size=100):
        self.predictor = predictor
        self.data = data
        self.size = size
        self.guesses, self.truths, self.errors, self.sles = [], [], [], []
        self.colors = []

    def color_for(self, error, truth):
        if error/truth < 0.2: return "green"
        elif error/truth < 0.4: return "orange"
        return "red"

    def run(self):
        for i in range(self.size):
            item = self.data[i]
            guess = self.predictor(item['description'])
            truth = item['price']
            error = abs(guess - truth)
            sle = (math.log(truth+1) - math.log(guess+1))**2
            
            self.guesses.append(guess)
            self.truths.append(truth)
            self.errors.append(error)
            self.sles.append(sle)
            self.colors.append(self.color_for(error, truth))
            
        self.report()

    def report(self):
        avg_error = sum(self.errors)/self.size
        rmsle = math.sqrt(sum(self.sles)/self.size)
        
        plt.figure(figsize=(10, 6))
        plt.scatter(self.truths, self.guesses, c=self.colors, s=10)
        plt.plot([0, max(self.truths)], [0, max(self.truths)], "k--", alpha=0.5)
        plt.title(f"Accuracy: ${avg_error:.2f} Avg Error | RMSLE: {rmsle:.4f}")
        plt.xlabel("Truth")
        plt.ylabel("Model Guess")
        plt.show()

## 2. RAG-Shot Inference Logic

Combining vector search with the fine-tuned model.

In [ ]:
def predict_with_rag(description, chroma_db, model, tokenizer):
    # 1. Retrieve 2 similar examples
    docs = chroma_db.similarity_search(description, k=2)
    shots = "\n\n".join([f"Description: {d.page_content}\nPrice: ${d.metadata['price']}" for d in docs])
    
    # 2. Build Shot-Prompt
    prompt = f"Examples:\n{shots}\n\nEstimate price for:\n{description}\nPrice is: $"
    
    # 3. Model Inference
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=5)
    raw_out = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract price using regex
    match = re.findall(r"\d+\.?\d*", raw_out.split("Price is: $")[-1])
    return float(match[0]) if match else 0.0